In [1]:
%load_ext cudf.pandas
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle

Enabled rmm statistics


In [3]:
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
start = time.time()

In [4]:
passmark = 40

In [5]:
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")

In [6]:
### cell 1 ###

factor = 1000
df = pd.concat([df]*factor)
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 1000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count    Dtype
---  ------                       --------------    -----
 0   gender                       1000000 non-null  object
 1   race/ethnicity               1000000 non-null  object
 2   parental level of education  1000000 non-null  object
 3   lunch                        1000000 non-null  object
 4   test preparation course      1000000 non-null  object
 5   math score                   1000000 non-null  object
 6   reading score                1000000 non-null  object
 7   writing score                1000000 non-null  object
dtypes: object(8)
memory usage: 83.8+ MB


In [7]:
### cell 2 ###

df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [8]:
### cell 3 ###

print (df.shape)

(1000000, 8)


In [9]:
### cell 4 ###

df.describe()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
unique,2,5,6,2,2,81,72,77
top,female,group C,some college,standard,none,65,72,74
freq,518000,319000,226000,645000,642000,36000,34000,35000


In [10]:
### cell 5 ###

df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [11]:
### cell 6 ###

df['math score'] = pd.to_numeric(df['math score'], errors='coerce')
df['Math_PassStatus'] = np.where(df['math score']<passmark, 'F', 'P')
df.Math_PassStatus.value_counts()

Math_PassStatus
P    960000
F     40000
Name: count, dtype: int64

In [12]:
### cell 7 ###

df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')
df['Reading_PassStatus'] = np.where(df['reading score']<passmark, 'F', 'P')
df.Reading_PassStatus.value_counts()

Reading_PassStatus
P    974000
F     26000
Name: count, dtype: int64

In [13]:
### cell 8 ###

df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')
df['Writing_PassStatus'] = np.where(df['writing score']<passmark, 'F', 'P')
df.Writing_PassStatus.value_counts()

Writing_PassStatus
P    968000
F     32000
Name: count, dtype: int64

In [14]:
### cell 9 ###

df['OverAll_PassStatus'] = df.apply(lambda x : 'F' if x['Math_PassStatus'] == 'F' or 
                                    x['Reading_PassStatus'] == 'F' or x['Writing_PassStatus'] == 'F' else 'P', axis =1)

df.OverAll_PassStatus.value_counts()

OverAll_PassStatus
P    949000
F     51000
Name: count, dtype: int64

In [15]:
### cell 10 ###

df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

In [16]:
### cell 11 ###

def GetGrade(Percentage, OverAll_PassStatus):
    if ( OverAll_PassStatus == 'F'):
        return 'F'    
    if ( Percentage >= 80 ):
        return 'A'
    if ( Percentage >= 70):
        return 'B'
    if ( Percentage >= 60):
        return 'C'
    if ( Percentage >= 50):
        return 'D'
    if ( Percentage >= 40):
        return 'E'
    else: 
        return 'F'

df['Grade'] = df.apply(lambda x : GetGrade(x['Percentage'], x['OverAll_PassStatus']), axis=1)

df.Grade.value_counts()

Grade
B    261000
C    256000
A    198000
D    178000
E     56000
F     51000
Name: count, dtype: int64

In [17]:
end = time.time()
print("Total execution time", end - start)

Total execution time 33.891117572784424
